In [1]:

"""
train_streaming.py

Full streaming training pipeline optimized for large datasets (~40GB) on Kaggle / P100:
- Streams images from disk (or webdataset tar shards)
- AMP (mixed precision)
- Gradient checkpointing (optional, depends on model)
- Gradient accumulation
- EarlyStopping + ReduceLROnPlateau
- Checkpoint manager keeping best N checkpoints
- Post-training pruning
- Training/validation curves visualization (matplotlib)
"""

import os
import math
import random
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm

# AMP
from torch.cuda.amp import autocast, GradScaler

# pruning
import torch.nn.utils.prune as prune

# plotting
import matplotlib.pyplot as plt


# ---------------------------
# CONFIG
# ---------------------------
class CFG:
    # Data
    meta_csv = "/kaggle/input/metadata.csv"   # CSV with columns: image_path,label
    data_root = "/kaggle/input/images"            # base folder if image_path relative
    use_webdataset = False                        # set True if you have tar shards and webdataset installed
    webdataset_pattern = "/kaggle/input/shards/shard-%06d.tar"  # pattern or 'shards-{00000..00010}.tar'
    # Model / training
    model_name = "resnet50"   # use resnet50 to save VRAM; change to resnet101 if you want (resnet101 heavier)
    img_size = 224
    batch_size = 64           # per step batch (use gradient_accumulation to emulate larger)
    gradient_accumulation_steps = 2  # to emulate effective batch_size = batch_size * accum
    num_workers = 6
    persistent_workers = True
    epochs = 30
    lr = 3e-4
    weight_decay = 1e-4
    patience = 6              # early stopping patience (on val loss)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint_dir = "./checkpoints"
    max_keep = 3
    prune_amount = 0.2
    seed = 42
    print_freq = 50
    use_gradient_checkpointing = False  # resnet supports checkpoint API in some libs; enabled if desired
    use_prefetch = True         # prefetch to GPU (simple utility)
    save_plots = True


In [ ]:

# ---------------------------
# Utilities
# ---------------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    import numpy as np
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(CFG.seed)
os.makedirs(CFG.checkpoint_dir, exist_ok=True)

# ---------------------------
# Streaming dataset (manifest-based)
# ---------------------------
import csv

class StreamingImageDataset(IterableDataset):
    """
    IterableDataset that streams images from a CSV manifest (image_path,label).
    Does NOT load entire list into memory if manifest is huge (it iterates file).
    It will optionally shuffle by reading file into small buffer (limited reservoir).
    """
    def __init__(self, manifest_path: str, data_root: str = ".", transform=None, shuffle: bool = True, buffer_size: int = 10000):
        self.manifest_path = manifest_path
        self.data_root = data_root
        self.transform = transform
        self.shuffle = shuffle
        self.buffer_size = buffer_size

    def __iter__(self):
        # We'll implement a streaming generator with reservoir-style small buffer shuffle
        # Read manifest line by line; keep a buffer of `buffer_size` items and yield randomly from it.
        buffer = []
        with open(self.manifest_path, "r", newline='') as f:
            reader = csv.reader(f)
            header = next(reader, None)
            # try detect header
            has_header = False
            if header and ("image" in ",".join(header).lower() or "label" in ",".join(header).lower()):
                # assume header row, skip
                has_header = True
            if has_header:
                # we already advanced one row as header; restart file cursor properly
                f.seek(0)
                next(reader)  # skip header

            for row in reader:
                if len(row) < 2:
                    continue
                img_path, label = row[0], row[1]
                full_path = img_path if os.path.isabs(img_path) else os.path.join(self.data_root, img_path)
                buffer.append( (full_path, int(label)) )
                if len(buffer) >= self.buffer_size:
                    if self.shuffle:
                        random.shuffle(buffer)
                    for item in buffer:
                        yield self._load_item(item)
                    buffer = []
            # flush remainder
            if buffer:
                if self.shuffle:
                    random.shuffle(buffer)
                for item in buffer:
                    yield self._load_item(item)

    def _load_item(self, item_tuple):
        path, label = item_tuple
        # load lazily from disk
        try:
            with Image.open(path) as img:
                img = img.convert("RGB")
                if self.transform:
                    img = self.transform(img)
        except Exception as e:
            # if corrupted image: return a black image
            img = Image.new("RGB", (CFG.img_size, CFG.img_size))
            if self.transform:
                img = self.transform(img)
        return img, label


In [ ]:

# ---------------------------
# Optional: WebDataset supplier
# ---------------------------
def make_webdataset_loader(pattern: str, batch_size: int, transform, num_workers: int=4, shuffle_shards: bool=True):
    if not HAS_WEBDATASET:
        raise ImportError("webdataset not installed. pip install webdataset")
    # pattern could be like "shards/shard-{000000..000100}.tar" or a list
    dataset = wds.WebDataset(pattern).decode("pil").to_tuple("jpg", "cls")
    # apply transforms
    dataset = dataset.map_tuple(lambda img, lbl: (transform(img), int(lbl)))
    # batching
    dataset = dataset.batched(batch_size, partial=True)
    loader = dataset.dataloader(num_workers=num_workers)
    return loader

# ---------------------------
# Transforms
# ---------------------------
train_transforms = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.RandomResizedCrop(CFG.img_size, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.CenterCrop(CFG.img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ---------------------------
# Checkpoint Manager & EarlyStopping
# ---------------------------
class CheckpointManager:
    def __init__(self, dir_path: str, max_keep: int = 3):
        self.dir_path = dir_path
        self.max_keep = max_keep
        self.saved = []  # list of (path, val_loss)
        os.makedirs(self.dir_path, exist_ok=True)

    def save(self, model: nn.Module, optimizer: optim.Optimizer, epoch: int, val_loss: float, val_acc: Optional[float]=None):
        fname = f"ckpt_epoch{epoch:03d}_loss{val_loss:.4f}_acc{(val_acc or 0):.4f}.pt"
        path = os.path.join(self.dir_path, fname)
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc
        }, path)
        self.saved.append((path, val_loss))
        # keep sorted by val_loss (ascending)
        self.saved = sorted(self.saved, key=lambda x: x[1])[:self.max_keep]
        # remove files not in saved
        saved_paths = set(p for p, _ in self.saved)
        for f in os.listdir(self.dir_path):
            full = os.path.join(self.dir_path, f)
            if full not in saved_paths and f.startswith("ckpt_"):
                try:
                    os.remove(full)
                except Exception:
                    pass

class EarlyStopping:
    def __init__(self, patience: int = 6, min_delta: float = 1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = None
        self.counter = 0
        self.early_stop = False

    def step(self, metric: float):
        if self.best is None or metric + self.min_delta < self.best:
            self.best = metric
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return self.early_stop


In [ ]:

# ---------------------------
# Model creator (lightweight)
# ---------------------------
def create_model(num_classes: int, model_name: str = "resnet50", use_gradient_checkpointing: bool = False):
    if model_name == "resnet50":
        model = models.resnet50(pretrained=True)
    elif model_name == "resnet101":
        model = models.resnet101(pretrained=True)
    else:
        raise ValueError("Only resnet50/resnet101 supported in this template")

    # Replace classifier
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # optionally enable gradient checkpointing if supported (for some models there are built-in APIs)
    if use_gradient_checkpointing:
        # naive: wrap model with checkpoint_sequential to reduce activation memory.
        # Note: checkpoint_sequential expects sequential modules; using children() is a quick heuristic.
        try:
            model = torch.utils.checkpoint.checkpoint_sequential(list(model.children()), segments=2)
            # If wrapped, can't directly set .fc; so caution: in practice use timm or wrappers.
        except Exception:
            # fallback: skip
            pass

    return model

# ---------------------------
# Train / Eval loops
# ---------------------------
def evaluate(model, loader, device, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
    val_loss = running_loss / max(total, 1)
    val_acc = correct / max(total, 1)
    return val_loss, val_acc

def train_model_streaming(manifest_csv: str,
                          num_classes: int,
                          data_root: str = CFG.data_root,
                          img_size: int = CFG.img_size,
                          batch_size: int = CFG.batch_size,
                          epochs: int = CFG.epochs,
                          lr: float = CFG.lr,
                          weight_decay: float = CFG.weight_decay,
                          gradient_accumulation_steps: int = CFG.gradient_accumulation_steps,
                          device: torch.device = CFG.device,
                          checkpoint_dir: str = CFG.checkpoint_dir,
                          max_keep: int = CFG.max_keep,
                          prune_amount: float = CFG.prune_amount,
                          use_webdataset: bool = CFG.use_webdataset,
                          webdataset_pattern: str = CFG.webdataset_pattern,
                          num_workers: int = CFG.num_workers):
    # prepare dataset
    if use_webdataset:
        if not HAS_WEBDATASET:
            raise ImportError("webdataset requested but not installed.")
        train_loader = make_webdataset_loader(webdataset_pattern, batch_size, train_transforms, num_workers=num_workers)
        # Webdataset splitting for val is user's responsibility (or create separate pattern)
        # For simplicity, we won't implement val with webdataset here.
        raise NotImplementedError("Webdataset val loader not implemented in this template.")
    else:
        # We will split manifest into train/val by reading it once and writing two small manifest files
        # This avoids loading images into memory.
        train_manifest = os.path.join(checkpoint_dir, "train_manifest.csv")
        val_manifest = os.path.join(checkpoint_dir, "val_manifest.csv")
        # Create split (streaming-friendly): iterate through original manifest, assign to train/val by ratio
        train_ratio = 0.90
        with open(manifest_csv, "r", newline='') as f_in, \
             open(train_manifest, "w", newline='') as f_train, \
             open(val_manifest, "w", newline='') as f_val:
            reader = csv.reader(f_in)
            header = next(reader, None)
            # if header exists and non-numeric in second col, keep header out of splitting
            # We'll just write rows directly
            for row in reader:
                if len(row) < 2:
                    continue
                if random.random() < train_ratio:
                    f_train.write(",".join(row) + "\n")
                else:
                    f_val.write(",".join(row) + "\n")

        train_dataset = StreamingImageDataset(train_manifest, data_root, transform=train_transforms, shuffle=True, buffer_size=20000)
        val_dataset = StreamingImageDataset(val_manifest, data_root, transform=val_transforms, shuffle=False, buffer_size=2000)

        # DataLoader: for IterableDataset, set shuffle=False, and rely on dataset's internal shuffle
        train_loader = DataLoader(train_dataset,
                                  batch_size=batch_size,
                                  num_workers=num_workers,
                                  pin_memory=True,
                                  persistent_workers=CFG.persistent_workers)
        val_loader = DataLoader(val_dataset,
                                batch_size=batch_size,
                                num_workers=max(1, min(4, num_workers)),
                                pin_memory=True,
                                persistent_workers=False)

    # Model, optimizer, criterion
    model = create_model(num_classes=num_classes, model_name=CFG.model_name, use_gradient_checkpointing=CFG.use_gradient_checkpointing)
    model = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    ckpt_mgr = CheckpointManager(checkpoint_dir, max_keep=max_keep)
    early_stop = EarlyStopping(patience=CFG.patience)

    # Metrics history
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    global_step = 0
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_samples = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} - train", ncols=120)
        optimizer.zero_grad()

        for step, batch in enumerate(pbar):
            # WebDataset returns batches; our IterableDataset returns single samples or batches depending on DataLoader
            if isinstance(batch, tuple) and isinstance(batch[0], torch.Tensor):
                imgs, labels = batch
            else:
                # If batch is a list of tuples (unlikely), try to unpack
                imgs, labels = batch

            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels) / gradient_accumulation_steps

            scaler.scale(loss).backward()

            if (step + 1) % gradient_accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            bs = imgs.size(0)
            running_loss += loss.item() * bs * gradient_accumulation_steps
            running_samples += bs
            global_step += 1

            if global_step % CFG.print_freq == 0:
                pbar.set_postfix({
                    "loss": f"{running_loss / max(1, running_samples):.4f}",
                    "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
                })

        # End epoch training metrics
        train_loss_epoch = running_loss / max(1, running_samples)
        history["train_loss"].append(train_loss_epoch)

        # Validation
        print("Evaluating on validation set...")
        val_loss, val_acc = evaluate(model, val_loader, device, criterion)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch} done. TrainLoss={train_loss_epoch:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}")

        # Scheduler step
        scheduler.step(val_loss)

        # Checkpoint (save best)
        ckpt_mgr.save(model, optimizer, epoch, val_loss, val_acc)

        # Early stopping
        if early_stop.step(val_loss):
            print(f"Early stopping triggered at epoch {epoch}")
            break

    # Post-training: global pruning on Linear layers (fc)
    parameters_to_prune = []
    for name, module in model.named_modules():
        # prune Linear and Conv2d optionally (we prune Linear classifier here)
        if isinstance(module, nn.Linear):
            parameters_to_prune.append((module, "weight"))
    if parameters_to_prune:
        prune.global_unstructured(parameters_to_prune, pruning_method=prune.L1Unstructured, amount=prune_amount)
        print(f"Applied global pruning amount={prune_amount} to {len(parameters_to_prune)} modules.")
    final_path = os.path.join(checkpoint_dir, "model_final_pruned.pth")
    torch.save(model.state_dict(), final_path)
    print("Final model saved to:", final_path)

    # Visualization
    if CFG.save_plots:
        plot_path = os.path.join(checkpoint_dir, "training_curves.png")
        _plot_history(history, plot_path)
        print("Saved training curves to:", plot_path)

    return model, history


In [ ]:

# ---------------------------
# Plot utility
# ---------------------------
def _plot_history(history: dict, save_path: str):
    plt.style.use('seaborn-darkgrid')
    fig, ax1 = plt.subplots(figsize=(10,5))
    ax1.plot(history["train_loss"], label="train_loss")
    ax1.plot(history["val_loss"], label="val_loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper left")
    ax2 = ax1.twinx()
    ax2.plot(history["val_acc"], label="val_acc", linestyle="--")
    ax2.set_ylabel("Val Acc")
    ax2.legend(loc="upper right")
    plt.title("Training Curves")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)

# ---------------------------
# Example usage
# ---------------------------
if __name__ == "__main__":
    # Example: assume manifest.csv exists with lines: relative/path/to/img.jpg,0
    # Determine num_classes by scanning manifest labels (cheap pass)
    manifest = CFG.manifest_csv
    labels_set = set()
    with open(manifest, "r") as f:
        reader = csv.reader(f)
        _h = next(reader, None)
        for r in reader:
            if len(r) < 2: continue
            labels_set.add(int(r[1]))
    num_classes = max(labels_set) + 1 if labels_set else 2

    model, history = train_model_streaming(
        manifest_csv=manifest,
        num_classes=num_classes,
        data_root=CFG.data_root,
        img_size=CFG.img_size,
        batch_size=CFG.batch_size,
        epochs=CFG.epochs,
        lr=CFG.lr,
        gradient_accumulation_steps=CFG.gradient_accumulation_steps,
        device=CFG.device,
        checkpoint_dir=CFG.checkpoint_dir,
        max_keep=CFG.max_keep,
        prune_amount=CFG.prune_amount,
        use_webdataset=CFG.use_webdataset,
        webdataset_pattern=CFG.webdataset_pattern,
        num_workers=CFG.num_workers
    )

    print("Done.")
